In [1]:
import numpy as np
import pandas as pd
import pickle
import os
from datetime import datetime


In [2]:
# ── PATHS ──────────────────────────────────────────────
ROOT_DIR    = '/Users/ruben/Desktop/Thesis/TrainingData/final-data'
DYNAMIC_CSV  = f'{ROOT_DIR}/dynamic_features_full.csv'
STATIC_CSV   = f'{ROOT_DIR}/static_osm_features_full_expanded.csv'
WEALTH_CSV   = f'{ROOT_DIR}/dhs_wealth.csv'
OUTPUT_DIR   = f'{ROOT_DIR}/output'

In [3]:
# ── LOAD ───────────────────────────────────────────────
print("Loading datasets...")
df_dyn    = pd.read_csv(DYNAMIC_CSV)
df_stat   = pd.read_csv(STATIC_CSV)
df_wealth = pd.read_csv(WEALTH_CSV)

df_dyn['ClusterID']   = df_dyn['ClusterID'].astype(int)
df_stat['DHSCLUST']   = df_stat['DHSCLUST'].astype(int)
df_wealth['DHSCLUST'] = df_wealth['DHSCLUST'].astype(int)

print(f"Dynamic  : {df_dyn.shape}")
print(f"Static   : {df_stat.shape}")
print(f"Wealth   : {df_wealth.shape}")

Loading datasets...
Dynamic  : (4939, 4098)
Static   : (1247, 53)
Wealth   : (1247, 2)


In [4]:
# ── COVERAGE CHECK ─────────────────────────────────────
dyn_clusters    = set(df_dyn['ClusterID'].unique())
stat_clusters   = set(df_stat['DHSCLUST'].unique())
wealth_clusters = set(df_wealth['DHSCLUST'].unique())
common          = dyn_clusters & stat_clusters & wealth_clusters

print(f"\nCluster coverage:")
print(f"  Dynamic          : {len(dyn_clusters)}")
print(f"  Static OSM       : {len(stat_clusters)}")
print(f"  Wealth labels    : {len(wealth_clusters)}")
print(f"  All three common : {len(common)}")

missing_stat   = dyn_clusters - stat_clusters
missing_wealth = dyn_clusters - wealth_clusters
if missing_stat:
    print(f"  WARNING: {len(missing_stat)} clusters in "
          f"dynamic but not OSM: {sorted(missing_stat)[:10]}")
if missing_wealth:
    print(f"  WARNING: {len(missing_wealth)} clusters in "
          f"dynamic but not wealth: {sorted(missing_wealth)[:10]}")


Cluster coverage:
  Dynamic          : 1236
  Static OSM       : 1247
  Wealth labels    : 1247
  All three common : 1236


In [5]:
# ── RESHAPE DYNAMIC: (N, 4, 4096) ─────────────────────
print("\nReshaping dynamic features into (N, 4, 4096)...")

# Only use clusters present in all three datasets
valid_for_reshape = sorted(common)
X_dynamic      = []
valid_clusters = []
skipped        = []

CNN_COLS = [c for c in df_dyn.columns
            if c.startswith('CNN_')]
print(f"CNN feature columns: {len(CNN_COLS)}")

for cid in valid_for_reshape:
    subset = df_dyn[
        df_dyn['ClusterID'] == cid
    ].sort_values('Quarter')

    if len(subset) == 4:
        feats = subset[CNN_COLS].values  # (4, 4096)
        X_dynamic.append(feats)
        valid_clusters.append(cid)
    else:
        skipped.append((cid, len(subset)))

X_dynamic = np.array(X_dynamic, dtype=np.float32)

print(f"Valid clusters (4 quarters each) : {len(valid_clusters)}")
print(f"Skipped (incomplete quarters)    : {len(skipped)}")
if skipped:
    print(f"  Sample skipped: {skipped[:5]}")
print(f"X_dynamic shape: {X_dynamic.shape}")


Reshaping dynamic features into (N, 4, 4096)...
CNN feature columns: 4096
Valid clusters (4 quarters each) : 1234
Skipped (incomplete quarters)    : 2
  Sample skipped: [(np.int64(1011), 1), (np.int64(1022), 2)]
X_dynamic shape: (1234, 4, 4096)


In [6]:
# ── ALIGN STATIC AND WEALTH ────────────────────────────
print("\nAligning static and wealth to valid clusters...")

# Log transform skewed features in static
log_cols = [
    'LU_Residential_m2', 'LU_Commercial_m2',
    'LU_Industrial_m2',  'LU_Agricultural_m2',
    'LU_Forest_m2',
    'POI_restaurant_Count', 'Total_POI_Count',
]
for col in log_cols:
    df_stat[col] = np.log1p(df_stat[col])

# Drop VIIRS_Median from static if it exists
# (VIIRS is kept as a static feature for the model)
static_feature_cols = [
    c for c in df_stat.columns if c != 'DHSCLUST']

df_stat_aligned = (
    df_stat[df_stat['DHSCLUST'].isin(valid_clusters)]
    .set_index('DHSCLUST')
    .reindex(valid_clusters)
)[static_feature_cols]

df_wealth_aligned = (
    df_wealth[df_wealth['DHSCLUST'].isin(valid_clusters)]
    .set_index('DHSCLUST')
    .reindex(valid_clusters)
)

X_static = df_stat_aligned.values.astype(np.float32)
y        = df_wealth_aligned['Wealth_Index'].values.astype(np.float32)

print(f"X_static shape : {X_static.shape}")
print(f"y shape        : {y.shape}")



Aligning static and wealth to valid clusters...
X_static shape : (1234, 52)
y shape        : (1234,)


In [7]:
# ── VALIDATE ───────────────────────────────────────────
print("\nValidating merged dataset...")

# Check for NaN in targets
nan_targets = np.isnan(y).sum()
if nan_targets > 0:
    print(f"WARNING: {nan_targets} NaN wealth values. Removing.")
    valid_mask     = ~np.isnan(y)
    X_dynamic      = X_dynamic[valid_mask]
    X_static       = X_static[valid_mask]
    y              = y[valid_mask]
    valid_clusters = [
        c for c, m in zip(valid_clusters, valid_mask) if m]
else:
    print("No NaN in wealth labels.")

# Check for NaN in static features
nan_static = np.isnan(X_static).sum()
if nan_static > 0:
    print(f"WARNING: {nan_static} NaN values in static features.")
    print("Filling with column means.")
    col_means = np.nanmean(X_static, axis=0)
    for j in range(X_static.shape[1]):
        mask = np.isnan(X_static[:, j])
        X_static[mask, j] = col_means[j]
else:
    print("No NaN in static features.")

# Check for NaN in dynamic features
nan_dynamic = np.isnan(X_dynamic).sum()
if nan_dynamic > 0:
    print(f"WARNING: {nan_dynamic} NaN in dynamic features.")
    X_dynamic = np.nan_to_num(X_dynamic, nan=0.0)
else:
    print("No NaN in dynamic features.")

print(f"\nFinal dataset:")
print(f"  X_dynamic : {X_dynamic.shape}")
print(f"  X_static  : {X_static.shape}")
print(f"  y         : {y.shape}")
print(f"  Clusters  : {len(valid_clusters)}")
print(f"  y range   : [{y.min():.4f}, {y.max():.4f}]")
print(f"  y mean    : {y.mean():.4f}")
print(f"  y std     : {y.std():.4f}")


Validating merged dataset...
No NaN in wealth labels.
No NaN in static features.
No NaN in dynamic features.

Final dataset:
  X_dynamic : (1234, 4, 4096)
  X_static  : (1234, 52)
  y         : (1234,)
  Clusters  : 1234
  y range   : [-2.3689, 1.8740]
  y mean    : 0.0473
  y std     : 0.6879


In [8]:
# ── SAVE ───────────────────────────────────────────────
print(f"\nSaving numpy arrays to {OUTPUT_DIR}...")

np.save(f'{OUTPUT_DIR}/X_dynamic.npy',   X_dynamic)
np.save(f'{OUTPUT_DIR}/X_static_expanded.npy',    X_static)
np.save(f'{OUTPUT_DIR}/y_wealth.npy',    y)
np.save(f'{OUTPUT_DIR}/cluster_ids.npy',
        np.array(valid_clusters))

# Save static feature names for SHAP later
static_feature_names = list(df_stat_aligned.columns)
with open(f'{OUTPUT_DIR}/static_feature_names.txt', 'w') as f:
    f.write('\n'.join(static_feature_names))

print(f"  X_dynamic.npy        "
      f"({X_dynamic.nbytes / 1e6:.1f} MB)")
print(f"  X_static.npy         "
      f"({X_static.nbytes / 1e6:.1f} MB)")
print(f"  y_wealth.npy")
print(f"  cluster_ids.npy")
print(f"  static_feature_names.txt")


Saving numpy arrays to /Users/ruben/Desktop/Thesis/TrainingData/final-data/output...
  X_dynamic.npy        (80.9 MB)
  X_static.npy         (0.3 MB)
  y_wealth.npy
  cluster_ids.npy
  static_feature_names.txt


In [9]:
# ── FINAL SUMMARY ──────────────────────────────────────
print(f"\n{'='*50}")
print("MERGE COMPLETE")
print(f"{'='*50}")
print(f"Clusters in LSTM training : {len(valid_clusters)}")
print(f"Dynamic feature shape     : {X_dynamic.shape}")
print(f"Static feature shape      : {X_static.shape}")
print(f"Static feature names      : {static_feature_names}")
print(f"\nReady for LSTM training.")


MERGE COMPLETE
Clusters in LSTM training : 1234
Dynamic feature shape     : (1234, 4, 4096)
Static feature shape      : (1234, 52)
Static feature names      : ['Total_Road_Length', 'Main_Roads_Length', 'Secondary_Roads_Length', 'Local_Roads_Length', 'Tracks_Length', 'Total_Bldg_Count', 'Total_Bldg_Area', 'Mean_Bldg_Area', 'Total_Bldg_Proportion', 'Bldg_Density_per_km2', 'Bldg_residential_Count', 'Bldg_residential_TotalArea', 'Bldg_residential_MeanArea', 'Bldg_residential_Proportion', 'Bldg_commercial_Count', 'Bldg_commercial_TotalArea', 'Bldg_commercial_MeanArea', 'Bldg_commercial_Proportion', 'Bldg_industrial_Count', 'Bldg_industrial_TotalArea', 'Bldg_industrial_MeanArea', 'Bldg_industrial_Proportion', 'Bldg_school_Count', 'Bldg_school_TotalArea', 'Bldg_school_MeanArea', 'Bldg_school_Proportion', 'Bldg_hospital_Count', 'Bldg_hospital_TotalArea', 'Bldg_hospital_MeanArea', 'Bldg_hospital_Proportion', 'Total_POI_Count', 'POI_bank_Count', 'POI_atm_Count', 'POI_hotel_Count', 'POI_fast_foo